# 🔬 Tracking Every Cell in a Developing Embryo — Explained

*A plain-language walkthrough of the **Biohub Cell Tracking During Development** problem: what it is, what we're trying to do, how our pipeline works, and the investigation that shaped our approach.*

> **Who is this for?** Anyone — no biology or machine-learning background assumed. We build up from "what is this even about" to the specific engineering decisions, and the one discovery that mattered most.

## 1. What is this project, really?

Every complex living thing — a fly, a fish, a person — starts as a **single cell**. That cell splits into two, those split into four, and on and on, until there are billions of cells organized into a body. One of the deepest questions in biology is *how* that happens: which cell becomes which, when cells divide, and how they move and arrange themselves into tissues and organs.

To study it, scientists film a developing embryo under a microscope. Not a flat photo — a **3D scan captured repeatedly over time**. Think of it as a movie where every single "frame" is a complete 3D volume, and inside that volume are thousands of glowing cell nuclei, drifting and dividing.

The dream is to reconstruct the **complete family tree of every cell**: its full history from one moment to the next, including every division. Biologists call this a *cell lineage*.

**The catch:** a single movie can contain **thousands of cells across thousands of time points**. Tracing every one by hand is effectively impossible. So the question this competition asks is simple to state and hard to answer:

> **Can a computer reconstruct the cell family tree automatically — and accurately?**

## 2. What exactly are we trying to produce?

Given the 3D movie, our program has to output a **graph** — a network of dots and lines that captures the family tree:

- a **node** (dot) for every cell in every frame — *where each cell is*,
- an **edge** (line) linking a cell to **itself in the next frame** — *who became who*,
- and the **branch points** where one cell **divides** into two daughters.

Get those links right across the whole movie and you've reconstructed the lineage. The figure below shows a tiny toy example: time runs left to right, each dot is a cell at one moment, lines follow a cell forward, and the fork is a division.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({"figure.dpi":120,"font.size":11,"axes.spines.top":False,"axes.spines.right":False})
TEAL,CORAL,GREEN,AMBER,INK,MUTE = "#0f9d90","#d94950","#2f9e63","#b9791a","#1a1d24","#8a93a3"

fig,ax=plt.subplots(figsize=(8,3.4))
# a toy lineage: one cell divides at t=3, both daughters continue
def track(xs,ys,c=TEAL): ax.plot(xs,ys,"-o",color=c,ms=8,lw=2.2,mfc="white",mec=c,mew=2)
track([0,1,2,3],[0,0.1,0.0,0.05])                       # parent
track([3,4,5,6],[0.05,0.55,0.7,0.75],GREEN)             # daughter 1
track([3,4,5,6],[0.05,-0.5,-0.65,-0.7],GREEN)           # daughter 2
ax.plot(3,0.05,"o",ms=14,mfc=AMBER,mec=AMBER)           # division event
ax.annotate("a cell divides",(3,0.05),(3.1,0.28),color=AMBER,fontsize=10,
            arrowprops=dict(arrowstyle="->",color=AMBER))
ax.annotate("one cell,\nfollowed over time",(0.5,0.05),(0.2,-0.55),color=MUTE,fontsize=9.5,
            arrowprops=dict(arrowstyle="->",color=MUTE))
ax.set_xlabel("time  (frame →)"); ax.set_yticks([]); ax.set_xticks(range(7))
ax.set_title("A cell lineage = the family tree we must reconstruct",loc="left",fontweight="bold")
ax.set_ylim(-1,1.1); plt.tight_layout(); plt.show()

## 3. How is a solution graded?

You might expect the score to be about *finding the cells*. It mostly isn't. The score has two parts:

- **~90% of your grade: the links.** Did you correctly connect each cell to itself in the next frame? This is measured with an *edge-Jaccard* — essentially, of all the true "who-became-who" links, what fraction did you get right (without inventing false ones)?
- **~10% of your grade: the divisions.** A small bonus for correctly catching the rare moments a cell splits in two.

A predicted cell "counts" as matching a real one if it lands within **7 micrometres** of it. And crucially, the ground-truth labels are **sparse** — only a subset of cells is labelled — so the metric rewards *linking the labelled cells correctly* far more than detecting every possible blob.

**The one-sentence takeaway:** *this is a connect-the-dots problem, not a find-the-dots problem.* That fact drove everything we did.

## 4. Our approach — a four-stage pipeline

We build on a strong open-source pipeline. It isn't a single model; it's an assembly line, where each stage feeds the next:

1. **Detect** — a 3D neural network scans each volume and finds the cell nuclei. (We also average its predictions over 8 rotated/flipped versions of the image — "test-time augmentation" — for a steadier result.)
2. **Rank** — a second "transformer" model looks at each cell and each candidate cell in the *next* frame and scores how likely they are the same cell.
3. **Solve** — an optimizer (an *integer linear program*) picks a globally consistent set of links: each cell continues to at most one cell, except where a division is allowed.
4. **Repair** — hand-crafted "graph surgery" cleans up the result: bridging short gaps, re-linking based on motion, removing implausible tracks, smoothing.

The figure shows the flow. The important thing to notice: **the detector is fixed and already excellent. Almost all of the tunable knobs live in stage 4.**

In [ ]:
fig,ax=plt.subplots(figsize=(9,2.2)); ax.axis("off")
stages=[("1 · Detect","3D U-Net + TTA\nfind the cells",TEAL),
        ("2 · Rank","transformer scores\nevery link",TEAL),
        ("3 · Solve","optimizer picks a\nconsistent graph",TEAL),
        ("4 · Repair","graph surgery:\ngaps · motion · filters",GREEN)]
for i,(t,s,c) in enumerate(stages):
    x=i*2.4
    ax.add_patch(plt.Rectangle((x,0),2.05,1.3,fc="white",ec=c,lw=2,zorder=2,
                 alpha=1 if c==GREEN else 0.9))
    ax.text(x+1.02,0.95,t,ha="center",fontweight="bold",color=c if c==GREEN else INK,fontsize=11,zorder=3)
    ax.text(x+1.02,0.42,s,ha="center",color=MUTE,fontsize=8.6,zorder=3)
    if i<3: ax.annotate("",(x+2.35,0.65),(x+2.05,0.65),arrowprops=dict(arrowstyle="-|>",color=MUTE,lw=1.6))
ax.text(7.62,-0.35,"← almost all of our tuning happens here",color=GREEN,fontsize=9.5,ha="right",fontweight="bold")
ax.set_xlim(-0.2,9.4); ax.set_ylim(-0.6,1.5); plt.tight_layout(); plt.show()

## 5. The investigation: where does a 0.90 pipeline lose its points?

Almost everyone in the competition starts from roughly this same base and lands near a score of **0.90**. Dozens of teams then sit there twiddling stage-4 knobs. To actually improve, we needed to know *where* the score leaks — not guess. So instead of blind tuning, we traced **every scored error back to its cause.** What follows is that detective story, in five steps.

### Step 1 — It's a linking problem, not a detection problem

We split every *missed* link into two kinds:
- **Detection-limited** — a cell was never found, so the link was impossible.
- **Linking-limited** — both cells *were* found; we simply failed to connect them.

The result was lopsided: **93% of our errors are linking mistakes among cells that were correctly detected.** Detection is basically solved; the whole crowd tuning detection thresholds was polishing the wrong 7%. And divisions? So rare in this data that the 0.1× bonus is a near-dead dial. **The lever is the linker.**

In [ ]:
fig,ax=plt.subplots(figsize=(7,1.5))
ax.barh([0],[93],color=TEAL,height=0.6); ax.barh([0],[7],left=[93],color=MUTE,height=0.6,alpha=0.5)
ax.text(46,0,"93%  linking errors (cells found, not connected)",ha="center",va="center",color="white",fontweight="bold",fontsize=11)
ax.text(96.5,0,"7% detection",ha="center",va="center",color=INK,fontsize=8.5)
ax.axis("off"); ax.set_xlim(0,100)
ax.set_title("Where the 0.90 pipeline actually loses points",loc="left",fontweight="bold")
plt.tight_layout(); plt.show()

### Step 2 — The linker gets *robbed* by fast-moving cells

Which links get missed? We compared missed links to correct ones on two suspects — **crowding** (too many nearby cells to tell apart) and **speed** (cells that move a lot). The verdict was clean: it's **not** crowding (cells are actually sparse) — it's **fast lateral motion**.

Here's the failure, and it's sneaky. A cell that jumps ~6µm sideways in one frame has its true partner *far away* next frame — but there's often another, unrelated blob *nearby*. The linker, biased toward "nearest," grabs the wrong nearby blob and abandons the true far partner. **One bad decision creates two errors at once:** a false link *and* a missed one. About 80% of our misses share this exact shape.

We then proved it can't be patched downstream: predicting where the cell "should" go from its recent motion pointed at the right cell only **1 in 28 times**, and — the decisive test — the ranking model gives the correct far link a probability of essentially **zero**. It was never even a candidate. *The model itself is blind to these associations.*

In [ ]:
fig,ax=plt.subplots(figsize=(8,3.2)); ax.axis("off")
ax.axvline(3.0,color=MUTE,ls="--",lw=1,alpha=0.6)
ax.text(1.0,2.35,"frame t",color=MUTE,ha="center"); ax.text(4.2,2.35,"frame t+1",color=MUTE,ha="center")
ax.plot(1.0,1.0,"o",ms=26,mfc="#d7f0ec",mec=TEAL,mew=2.5); ax.text(1.0,0.55,"the cell",ha="center",color=INK,fontsize=9)
ax.plot(4.3,0.7,"o",ms=24,mfc="#f7dcdd",mec=CORAL,mew=2.5); ax.text(4.3,0.28,"nearby blob",ha="center",color=CORAL,fontsize=9)
ax.plot(5.0,1.85,"o",ms=24,mfc="#d8efe1",mec=GREEN,mew=2.5); ax.text(5.0,2.2,"true partner",ha="center",color=GREEN,fontsize=9)
ax.annotate("",(4.15,0.72),(1.2,0.98),arrowprops=dict(arrowstyle="-|>",color=CORAL,lw=2.6))
ax.text(2.5,0.5,"✗ what it does (grabs nearest)",color=CORAL,ha="center",fontsize=9)
ax.annotate("",(4.85,1.82),(1.2,1.05),arrowprops=dict(arrowstyle="-|>",color=GREEN,lw=2,ls="--"))
ax.text(3.1,1.72,"✓ correct — a 6µm jump",color=GREEN,ha="center",fontsize=9)
ax.set_xlim(0,6); ax.set_ylim(0,2.6)
ax.set_title("The 'fast-motion steal' — the core failure",loc="left",fontweight="bold")
plt.tight_layout(); plt.show()

### Step 3 — So we tried to *teach* the model (a GPU experiment)

If the model never learned that cells *can* move far in one step, we'd show it examples. We retrained the linker with **frame-skip augmentation**: feed it `frame t → frame t+2` (and `t+3`) pairs, but *label them as if they were adjacent frames*. Now a normal-looking "next frame" carries a large spatial jump — exactly the 6–8µm examples the model never saw.

**It genuinely worked** — the retrained model recovered **40–48%** of the specific fast-motion links it used to miss. But there was a tax: teaching it that "far links are OK" also caused it to **break some links it used to get right.** On our validation, the gains and the damage roughly cancelled.

Then came the twist. On the actual leaderboard, the retrained models scored **0.889** and **0.895** — *below* the 0.900 base — even though our validation said they were better. Worse, the checkpoint with the **higher** validation score did **worse** on the real test. **Something was wrong with how we were measuring.**

### Step 4 — The real bug wasn't the model. It was our ruler. 📏

We'd been checking our work on 5 held-out videos. But all 5 were the same embryo strain — call it **strain A**. It turns out the training data is **64% a second strain** — **strain B** — and the *test set is half strain B*. And the strain-B videos are **5–10× denser** (thousands of cells instead of hundreds) — exactly where the frame-skip over-linking did the most damage.

In other words: **we'd been grading ourselves on a ruler that ignored the harder half of the test.** That's how a model can look better in validation and be worse in reality. The fix cost zero GPU: rebuild the held-out set to **mirror the test** — half strain A, half strain B, a range of sizes.

In [ ]:
import numpy as np
fig,ax=plt.subplots(figsize=(8,3)); 
groups=["Train set","Test set","Old validation\n(broken)","New validation\n(fixed)"]
A=np.array([71,2,5,5]); B=np.array([128,2,0,5])
Ap=A/(A+B)*100; Bp=B/(A+B)*100
x=np.arange(len(groups))
ax.bar(x,Ap,color=TEAL,label="strain A (sparse)")
ax.bar(x,Bp,bottom=Ap,color=CORAL,label="strain B (dense)")
for i in range(len(groups)):
    if Bp[i]>0: ax.text(i,Ap[i]+Bp[i]/2,f"{int(Bp[i])}%",ha="center",va="center",color="white",fontsize=9,fontweight="bold")
    if Ap[i]>0: ax.text(i,Ap[i]/2,f"{int(Ap[i])}%",ha="center",va="center",color="white",fontsize=9,fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(groups,fontsize=9.5); ax.set_yticks([]); 
ax.spines["left"].set_visible(False)
ax.set_title("The blind spot: our old validation had 0% of the dense strain B —\nyet the test set is half strain B",
             loc="left",fontweight="bold",fontsize=11)
ax.legend(loc="upper right",frameon=False,fontsize=9); plt.tight_layout(); plt.show()

### Step 5 — The honest ruler instantly found a config we'd been hiding from ourselves

With a validation set that finally reflected the test, we ran **one** prediction and swept every post-processing combination on top of it (cheap — the expensive part is predicting; scoring configs is free). A config we would *never* have shipped — because on the old broken ruler it looked identical to our champion — pulled clearly ahead.

Two of our knobs, **keep-fallback** (recover missed links) and **gap-confirm** (remove false links), had looked like *substitutes* on the strain-A-only ruler. On the dense strain-B videos they turn out to be **complementary** — one fixes recall, the other fixes precision — and *together* they beat everything we'd ever submitted by **+0.010** on the honest metric.

In [ ]:
fig,ax=plt.subplots(figsize=(8,3))
labels=["keepfb + gap-confirm","keep-fallback only","competitor '0.902' config","gap-confirm (old champion)","baseline"]
scores=[0.8717,0.8681,0.8622,0.8617,0.8577]
cols=[GREEN,TEAL,MUTE,MUTE,MUTE]
y=np.arange(len(labels))[::-1]
ax.barh(y,[s-0.855 for s in scores],left=0.855,color=cols,height=0.62)
for yi,s,l in zip(y,scores,labels):
    ax.text(0.8555,yi,f"  {l}",va="center",ha="left",fontsize=9.5,
            color="white" if l.startswith("keepfb") else INK,
            fontweight="bold" if l.startswith("keepfb") else "normal")
    ax.text(s+0.0004,yi,f"{s:.4f}",va="center",fontsize=9,fontfamily="monospace",color=INK)
ax.set_xlim(0.855,0.876); ax.set_yticks([]); ax.spines["left"].set_visible(False)
ax.set_title("Config ranking on the fixed validation — the winner was hiding in plain sight",
             loc="left",fontweight="bold",fontsize=11)
ax.set_xlabel("representative-CV score (higher = better)"); plt.tight_layout(); plt.show()

## 6. The lessons that outlast this competition

- **🔍 Trace the error before you tune.** One failure analysis redirected the entire effort — from detection (7% of the error) to linking (93%), and specifically to fast lateral motion. Blind knob-tuning would never have found it.
- **📏 A validation set that doesn't mirror the test is *worse than none* — it can rank a worse model higher.** Fixing ours (matching the strain mix) was the single highest-leverage move of the project, and it cost zero compute.
- **🧭 Validation is a compass, not a verdict.** Even a good validation number shrinks — and sometimes flips — on the real leaderboard. Every promising idea still earns a real submission; the number only tells you where to look.
- **⚙️ Optimize for the true bottleneck.** Our scarce resource was GPU time, not submissions. So we *predict once and sweep every config on top*, cache the intermediate results to iterate for free, and spend GPU only where an experiment truly needs it.

## 7. Where it stands, and what's open

| Submission | Leaderboard | Note |
|---|---|---|
| base blend | 0.900 | the crowd's starting point |
| gap-confirm | 0.901 | prior champion |
| + competitor tweaks | 0.902 | banked |
| frame-skip retrain (v5 / v4) | 0.889 / 0.895 | recovery real, but net-negative — and it exposed the broken ruler |
| **base + keep-fallback + gap-confirm** | *pending* | the honest ruler's pick — the shot that tests everything |

The pending submission is a two-in-one test: whether we have a new best, **and** whether our rebuilt validation is finally trustworthy enough to steer the rest of the climb.

And the retrain path isn't dead — the fast-motion recovery is *real*. The open challenge is keeping that recovery **without** the precision tax (for example, using *distillation* to preserve the links the model already gets right while it learns the new ones).

---
*Solution notes — Biohub Cell Tracking During Development. All numbers are from our own held-out-validation and public-leaderboard runs. The throughline: **the biggest win came from fixing how we measured, not from a fancier model.***